## Part 1: Data CLeaning & Validation
Ada Kam

In [213]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path
from IPython.display import display
import shutil

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
db_dir = root/'Lectures/Lecture 4'

### 1.1 Load and Inspect data

In [214]:
# Load ohlc_hw from data/data.db
database = db_dir/'data/data.db'
if not database.is_file():
    raise FileNotFoundError(f'No file found')
conn = sqlite3.connect(database)
ohlc_hw = pd.read_sql_query("SELECT * FROM ohlc_hw", conn, parse_dates=['ts'])

# Display basic statistics
display(ohlc_hw.info())
display(ohlc_hw.describe())

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[us]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  str           
dtypes: datetime64[us](1), float64(5), int64(1), str(1)
memory usage: 24.3 MB


None

,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292486,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


### Column meanings
|Column|Meaning|
|---|---|
|ts|Timestamp of data|
|open|First traded px of timestamp|
|high|Max px reached during timestamp|
|low|Min px reached during timestamp|
|close|Last traded px of timestamp|
|adj_close|Close px adjusted for splits/dvds|
|volume|Traded qty for timestamp|
|ticker|Asset symbol for identification|

In [215]:
# Identify Columns
display(pd.read_sql_query("PRAGMA table_info(ohlc_hw)", conn))

# Date range and list of tickers
display(pd.read_sql_query("SELECT MIN(ts) as start_dt, MAX(ts) as end_dt FROM ohlc_hw", conn))
raw_tickers=pd.read_sql_query("SELECT DISTINCT ticker FROM ohlc_hw", conn)
tickers=sorted(raw_tickers['ticker'].str.replace(r'<[^>]*>', '', regex=True).unique().tolist())
ohlc_hw['ticker_raw']=ohlc_hw['ticker']
ohlc_hw['ticker']=ohlc_hw['ticker'].str.replace(r'<[^>]*>', '', regex=True)
ohlc_hw['ticker']=ohlc_hw['ticker'].str.strip().str.upper()
print(tickers)

,cid,name,type,notnull,dflt_value,pk
0,0,ts,DATE,0,None,0
1,1,open,FLOAT,0,None,0
2,2,high,FLOAT,0,None,0
3,3,low,FLOAT,0,None,0
4,4,close,FLOAT,0,None,0
5,5,adj_close,FLOAT,0,None,0
6,6,volume,BIGINT,0,None,0
7,7,ticker,TEXT,0,None,0


,start_dt,end_dt
0,1962-01-02,2024-12-06


['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


### 1.2 Detect Data Quality Issues

In [216]:
# detect missing values in each column
ohlc_hw=ohlc_hw.drop_duplicates(subset=ohlc_hw.columns.drop('ticker_raw'))
def missing_values(data):
    id = ['ticker', 'ts']
    check_columns = [col for col in data.columns if col not in id]

    melt_data = data.melt(id_vars=id, value_vars=check_columns, var_name='missing_col', value_name='value')

    missing_rows=melt_data[melt_data['value'].isna()].copy()

    result = pd.DataFrame({
        'ticker': missing_rows['ticker'],
        'issue_type': 'Missing Data',
        'issue_date': missing_rows['ts'],
        'description': 'Missing ' + missing_rows['missing_col']
    })

    return result.reset_index(drop=True)


missing_values(ohlc_hw)


,ticker,issue_type,issue_date,description
0,KO,Missing Data,2024-04-16,Missing adj_close
1,NVDA,Missing Data,2018-06-12,Missing adj_close
2,HD,Missing Data,2013-04-16,Missing adj_close
3,PG,Missing Data,2015-03-20,Missing adj_close
4,JPM,Missing Data,2018-03-13,Missing adj_close
...,...,...,...,...
18369,MSFT,Missing Data,2012-09-25,Missing adj_close
18370,JNJ,Missing Data,2008-05-08,Missing adj_close
18371,NKE,Missing Data,2019-12-19,Missing adj_close
18372,BA,Missing Data,1976-07-09,Missing adj_close


In [217]:
# detect missing trading days
ohlc = pd.read_sql_query("SELECT * FROM ohlc", conn, parse_dates=['ts'])

def missing_dates(data):
    result = []
    for ticker, dates in data.groupby("ticker")["ts"]:
        start_dt, end_dt = dates.min(), dates.max()
        reference = ohlc.loc[ohlc["ticker"]==ticker, "ts"]
        reference = reference[(reference>=start_dt)&(reference<=end_dt)]
        missing = sorted(set(reference)-set(dates))

        if missing:
            for dt in missing:
                result.append({
                    'ticker': ticker,
                    'issue_type': "Missing Date",
                    'issue_date': dt,
                    'description': "Date Gap"
                })
            print(f"Ticker {ticker} is missing {len(missing)} trading days between {start_dt:%Y-%m-%d} and {end_dt:%Y-%m-%d}")

    return pd.DataFrame(result, columns=['ticker', 'issue_type','issue_date', 'description']).astype({'issue_date':'datetime64[ns]'})

display(missing_dates(ohlc_hw))
    

,ticker,issue_type,issue_date,description


In [218]:
# calculate daily returns
def daily_returns(data):
    data=data.sort_values(by=["ticker", "ts"])
    data["prev"]=data.groupby("ticker")["close"].shift(1)
    data["return"]=(data["close"]-data["prev"])/data["prev"]
    return data

display(ohlc_hw.head())

,ts,open,high,low,close,adj_close,volume,ticker,ticker_raw
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL,AAPL


In [219]:
def return_z(data):
    data_copy=daily_returns(data)
    returns = data_copy["return"].where(np.isfinite(data_copy["return"]))

    grouped = returns.groupby(data_copy["ticker"])
    stdev = grouped.transform(lambda x: x.std(ddof=1))
    return abs(returns-grouped.transform("mean"))/stdev.where(stdev>0)

def extreme_returns(data):
    data = data.reset_index(drop=True)
    z = return_z(data)
    z = z[z>3]
    above_three = data.loc[z.index]
    result=pd.DataFrame({
        "ticker": above_three['ticker'],
        "issue_type": "Extreme Returns",
        "issue_date": above_three['ts'],
        "description": z.round(2).astype(str)+" s.d. from mean"
    })

    return (result)

In [220]:
def adj_close_mask(data):
    sorted_data = data.sort_values(['ticker', 'ts'])
    factor = sorted_data['adj_close']/sorted_data['close']
    usual = factor.groupby(sorted_data['ticker']).transform(lambda f: f.rolling(window=11, center=True, min_periods=3).median())
    return ((factor/usual-1).abs() > 0.2).reindex(data.index)


def inconsistent_adj_close(data):
    error = data[adj_close_mask(data)]
    return pd.DataFrame({
        'ticker': error['ticker'],
        'issue_type': "Inconsistent Adj Close",
        'issue_date': error['ts'],
        'description': 'Extreme adj_close values'
    })

In [221]:
# flag high < close or low > close
def impossible_mask(data):
    return (data['high']<data['close']) | (data['low']>data['close'])

def impossible_data(data):
    mask = impossible_mask(data)
    error = data[mask]
    results=pd.DataFrame({
        'ticker': error["ticker"], 
        'issue_type': "Impossible Data",
        'issue_date': error['ts'],
        'description': "High/Low px out of range"
    })
    return results


display(impossible_data(ohlc_hw))

,ticker,issue_type,issue_date,description


In [222]:
# flag nonpositive or non-finite px
def non_positive_mask(data):
    cols = ['open', 'high', 'low','close', 'adj_close']
    prices = data[cols]
    return (prices<=0) | np.isinf(prices)

def non_positive(data):
    mask = non_positive_mask(data).any(axis=1)
    error = data[mask]
    results=pd.DataFrame({
            'ticker': error["ticker"], 
            'issue_type': "Non Positive Data",
            'issue_date': error['ts'],
            'description': "Negative/Infinite px"
        })
    return (results)

display(non_positive(ohlc_hw))

,ticker,issue_type,issue_date,description
99637,PG,Non Positive Data,1962-01-02,Negative/Infinite px
224790,MCD,Non Positive Data,1966-07-05,Negative/Infinite px
287343,CVX,Non Positive Data,1962-01-02,Negative/Infinite px
350808,JNJ,Non Positive Data,1962-01-02,Negative/Infinite px
364568,MRK,Non Positive Data,1962-01-02,Negative/Infinite px


In [223]:
# Verify low<=open<=high and low<=close<=high
def open_outside_mkt(data):
    return (data['low']<=data['open']) & (data['open']<=data['high']) & \
        (data['low']<=data['close']) & (data['close']<=data['high'])

def verify(data):
    cols = ['open', 'high','low', 'close']
    complete = data[cols].notna().all(axis=1)
    verified = open_outside_mkt(data)
    error = data[complete & ~verified]
    results=pd.DataFrame({
                'ticker': error["ticker"],
                'issue_type': "Verify Order",
                'issue_date': error['ts'],
                'description': "OHLC order violated"
            })
    
    return (results)

display(verify(ohlc_hw))

,ticker,issue_type,issue_date,description
99637,PG,Verify Order,1962-01-02,OHLC order violated
224790,MCD,Verify Order,1966-07-05,OHLC order violated
287343,CVX,Verify Order,1962-01-02,OHLC order violated
350808,JNJ,Verify Order,1962-01-02,OHLC order violated
364568,MRK,Verify Order,1962-01-02,OHLC order violated


In [224]:
# check zero volume data
def check_zero_volume(data):
    error = data[data['volume']==0]
    results=pd.DataFrame({
                'ticker': error["ticker"], 
                'issue_type': "Zero Volume",
                'issue_date': error['ts'],
                'description': "Volume=0"
            })
    
    return (results)

check_zero_volume(ohlc_hw)

,ticker,issue_type,issue_date,description
8975,TRV,Zero Volume,1978-12-20,Volume=0
15571,WMT,Zero Volume,1975-05-01,Volume=0
16521,WMT,Zero Volume,1973-09-10,Volume=0
22251,JNJ,Zero Volume,1962-09-28,Volume=0
28137,WMT,Zero Volume,1974-04-10,Volume=0
...,...,...,...,...
381407,WMT,Zero Volume,1973-09-05,Volume=0
381720,NKE,Zero Volume,1981-08-10,Volume=0
384157,TRV,Zero Volume,1976-10-28,Volume=0
394476,WMT,Zero Volume,1974-01-16,Volume=0


In [225]:
# generate data quality report
def generate_report(data):
     rows = [
          missing_values(data),
          missing_dates(data),
          extreme_returns(data),
          inconsistent_adj_close(data),
          impossible_data(data),
          non_positive(data),
          verify(data),
          check_zero_volume(data)
     ]
     report = pd.concat(rows, axis=0, ignore_index=True)

     severity = {
          "Missing Data":"medium", # needs review but can be handled using prev data
          "Missing Date":"high", # gaps disrupt rolling calculations
          "Extreme Returns":"low", # needs review but can be legitimate
          "Inconsistent Adj Close": "medium",
          "Impossible Data":"medium", 
          "Non Positive Data":"medium",
          "Verify Order":"medium",
          "Zero Volume":"low",
     }

     report["severity"]=report["issue_type"].map(severity)

     return report


display(generate_report(ohlc_hw))

,ticker,issue_type,issue_date,description,severity
0,KO,Missing Data,2024-04-16,Missing adj_close,medium
1,NVDA,Missing Data,2018-06-12,Missing adj_close,medium
2,HD,Missing Data,2013-04-16,Missing adj_close,medium
3,PG,Missing Data,2015-03-20,Missing adj_close,medium
4,JPM,Missing Data,2018-03-13,Missing adj_close,medium
...,...,...,...,...,...
33932,WMT,Zero Volume,1973-09-05,Volume=0,low
33933,NKE,Zero Volume,1981-08-10,Volume=0,low
33934,TRV,Zero Volume,1976-10-28,Volume=0,low
33935,WMT,Zero Volume,1974-01-16,Volume=0,low


In [226]:
# Demo
# Test data: NVDA 2020
base = (ohlc_hw.loc[(ohlc_hw['ticker']=='NVDA') & (ohlc_hw['ts'].dt.year==2020),['ts','open','high','low','close','adj_close', 'volume', 'ticker']].sort_values('ts').reset_index(drop=True))
baseline_report = generate_report(base)
display(baseline_report)

# Inject faults
test = base.copy()
test = test.drop(test.index[3:6]).reset_index(drop=True) # missing dates
test.loc[3, 'adj_close']=np.nan # missing value
test.loc[5, ['close', 'high']]=test.loc[30,'close']*10 #extreme return
test.loc[18, ['low', 'open']]=test.loc[18, ['open', 'low']] # wrong order
test.loc[10, ['close']]+=1 # close > high
test.loc[20, ['open']]=-1 # non positive px
test.loc[25, ['volume']]=0 # volume = 0

test_report=generate_report(test)
display(test_report)

,ticker,issue_type,issue_date,description,severity
0,NVDA,Missing Data,2020-01-03,Missing adj_close,medium
1,NVDA,Missing Data,2020-02-12,Missing adj_close,medium
2,NVDA,Missing Data,2020-03-12,Missing adj_close,medium
3,NVDA,Missing Data,2020-03-17,Missing adj_close,medium
4,NVDA,Missing Data,2020-04-01,Missing adj_close,medium
5,NVDA,Missing Data,2020-08-26,Missing adj_close,medium
6,NVDA,Missing Data,2020-10-21,Missing adj_close,medium
7,NVDA,Missing Data,2020-11-05,Missing adj_close,medium
8,NVDA,Missing Data,2020-11-19,Missing adj_close,medium
9,NVDA,Missing Data,2020-11-25,Missing adj_close,medium


Ticker NVDA is missing 3 trading days between 2020-01-02 and 2020-12-31


,ticker,issue_type,issue_date,description,severity
0,NVDA,Missing Data,2020-01-03,Missing adj_close,medium
1,NVDA,Missing Data,2020-01-10,Missing adj_close,medium
2,NVDA,Missing Data,2020-02-12,Missing adj_close,medium
3,NVDA,Missing Data,2020-03-12,Missing adj_close,medium
4,NVDA,Missing Data,2020-03-17,Missing adj_close,medium
5,NVDA,Missing Data,2020-04-01,Missing adj_close,medium
6,NVDA,Missing Data,2020-08-26,Missing adj_close,medium
7,NVDA,Missing Data,2020-10-21,Missing adj_close,medium
8,NVDA,Missing Data,2020-11-05,Missing adj_close,medium
9,NVDA,Missing Data,2020-11-19,Missing adj_close,medium


### 1.3 Clean the Data

In [227]:
def handle_missing_values(data):
    data=data.sort_values(by=['ticker','ts']).reset_index(drop=True)

    fill_cols=['open', 'high', 'low', 'close']
    data=data.dropna(subset=fill_cols, how='all').reset_index(drop=True) # drop rows where all columns are empty
    data['price_cleaned']=data[fill_cols].isna().any(axis=1)
    data[fill_cols]=data.groupby('ticker')[fill_cols].ffill()

    data['volume_cleaned']=data['volume'].isna()
    data['volume']=data['volume'].fillna(0)

    data['adj_close_corrupt'] = adj_close_mask(data)
    data.loc[data['adj_close_corrupt'], 'adj_close'] = np.nan

    ref = ohlc[['ticker', 'ts', 'close', 'adj_close']].rename(columns={'close': 'ref_close', 'adj_close':'ref_adj_close'})
    data=data.merge(ref,on=['ticker', 'ts'], how='left')
    factor = data['ref_adj_close']/data['ref_close']
    to_fill = data['adj_close'].isna() & np.isfinite(factor) & (data['ref_close'] > 0)
    data.loc[to_fill, 'adj_close']=data.loc[to_fill, 'close']*factor[to_fill]
    data['adj_close_cleaned']=to_fill
    data=data.drop(columns=['ref_close', 'ref_adj_close'])

    unresolved = data[data['adj_close'].isna()]
    print(f"adj_close cleaned: {to_fill.sum()}, unresolved: {len(unresolved)}")
    print(f"{data['price_cleaned'].sum()} price data cleaned, {data['volume_cleaned'].sum()} volume data cleaned")

    return data


ohlc_hw_copy=handle_missing_values(ohlc_hw)


adj_close cleaned: 28796, unresolved: 0
0 price data cleaned, 0 volume data cleaned


In [228]:
def interpolate_rows(data, rows):
    data = data.sort_values(['ticker', 'ts']).copy()

    if not isinstance(rows, pd.Series):
        targets = pd.MultiIndex.from_tuples([(t, pd.Timestamp(d)) for t, d in rows])
        rows = pd.Series(pd.MultiIndex.from_frame(data[['ticker', 'ts']]).isin(targets), index=data.index)
    else:
        rows = rows.reindex(data.index, fill_value=False)

    by_ticker = data.groupby('ticker')
    mid_close = (by_ticker['close'].shift(1) + by_ticker['close'].shift(-1)) / 2
    mid_adj = (by_ticker['adj_close'].shift(1) + by_ticker['adj_close'].shift(-1)) / 2

    data.loc[rows, ['open', 'high', 'low', 'close']] = mid_close[rows]
    data.loc[rows, 'adj_close'] = mid_adj[rows]

    data['interpolated'] = data.get('interpolated', False) | rows
    
    print(f"{rows.sum()} rows interpolated from neighbouring days")
    return data
    

In [229]:
# Handle Outliers
def handle_outliers(data, interpolate=None):
    data = data.sort_values(['ticker', 'ts']).copy()
    prev_close = data.groupby('ticker')['close'].shift(1)
    fallback = prev_close.fillna(data['close'])
    cols = ['open', 'high', 'low', 'close']

    non_pos = non_positive_mask(data)
    imp = impossible_mask(data)
    out_mkt = ~open_outside_mkt(data)

    # map errors to target columns
    mask = {
        'open': non_pos['open'] | out_mkt,
        'high': non_pos['high'] | imp,
        'low': non_pos['low'] | imp,
        'close': non_pos['close'] | imp,
        'adj_close': non_pos['adj_close']
    }

    for col, col_mask in mask.items():
        source = fallback if col == 'open' else prev_close
        data.loc[col_mask, col] = source[col_mask]
    data['outlier_cleaned'] = non_pos.any(axis=1) | imp | out_mkt


    print(f"{(non_pos.sum().sum())} Non-positive values cleaned")
    print(f"{imp.sum()} Impossible OHLC px cleaned")
    print(f"{out_mkt.sum()} Wrong OHLC px order cleaned")

    # classify extreme returns
    z = return_z(data)
    extreme = z>3
    direction = np.sign(data.groupby('ticker')['close'].pct_change()).where(extreme)
    same_move = direction.groupby([data['ts'], direction]).transform('count')
    avg_volume = data.groupby('ticker')['volume'].transform(lambda v: v.shift(1).rolling(5).mean())

    data['extreme_category'] = np.select(
        [same_move>=5, data['volume']>1.5*avg_volume],
        ['market-wide', 'high-volume'],
        default='unexplained')
    data['extreme_category'] = data['extreme_category'].where(extreme)

    counts = data['extreme_category'].value_counts()
    print(f"{extreme.sum()} extreme returns: {counts.get('market-wide', 0)} market-wide, "
          f"{counts.get('high-volume', 0)} high-volume, {counts.get('unexplained', 0)} unexplained")

    above_ten = z>10
    review = (z>10) & (data['extreme_category']=='unexplained')
    print(f"{review.sum()} returns more than 10 s.d. from mean:")
    display(data.loc[review, ['ticker', 'ts', 'close', 'volume', 'extreme_category']].assign(
        prev_close=data.groupby('ticker')['close'].shift(1)[review],
        next_close=data.groupby('ticker')['close'].shift(-1)[review],
        z=z[review].round(2)))

    # interpolate confirmed data errors, given as [(ticker, 'YYYY-MM-DD'), ...]
    data = interpolate_rows(data, review & (data['volume']==0))

    return data

cleaned_df = handle_outliers(ohlc_hw_copy)

5 Non-positive values cleaned
0 Impossible OHLC px cleaned
5 Wrong OHLC px order cleaned
5056 extreme returns: 1418 market-wide, 2204 high-volume, 1434 unexplained
8 returns more than 10 s.d. from mean:


,ticker,ts,close,volume,extreme_category,prev_close,next_close,z
193606,JPM,2009-01-21,22.63000,136495312,unexplained,18.09000,23.10000,11.94
318332,TRV,1976-10-28,9.22000,0,unexplained,4.61000,4.69000,41.24
318333,TRV,1976-10-29,4.69000,79200,unexplained,9.22000,4.66000,20.30
318873,TRV,1978-12-20,8.03000,0,unexplained,4.02000,4.09000,41.14
318874,TRV,1978-12-21,4.09000,105600,unexplained,8.03000,4.09000,20.27
319538,TRV,1981-08-10,12.06000,0,unexplained,6.03000,6.03000,41.24
319539,TRV,1981-08-11,6.03000,24800,unexplained,12.06000,6.13000,20.66
331071,UNH,1987-05-04,0.19922,8681600,unexplained,0.15625,0.20703,10.98


3 rows interpolated from neighbouring days


In [230]:
FLAG_COLS = ['price_cleaned', 'volume_cleaned', 'adj_close_corrupt', 'adj_close_cleaned',
             'outlier_cleaned', 'interpolated', 'extreme_category']

def save(data):
    cols = ['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker', 'ticker_raw']
    data = data[cols + [c for c in FLAG_COLS if c in data.columns]].copy()
    data['ts'] = data['ts'].dt.strftime('%Y-%m-%d')

    working_db = Path.cwd() / 'data' / 'data.db'
    working_db.parent.mkdir(exist_ok=True)
    if not working_db.is_file():
        shutil.copy2(database, working_db)

    with sqlite3.connect(working_db) as save_conn:
        data.to_sql('ohlc_hw_cleaned', save_conn, if_exists='replace', index=False)
    print(f"saved {len(data)} rows to ohlc_hw_cleaned in {working_db}")


In [231]:
def clean_data(data):
    data = handle_missing_values(data)
    data = handle_outliers(data)
    save(data)
    return data

In [232]:
## Validation
before=generate_report(ohlc_hw)
clean_data(ohlc_hw)
validate = pd.read_sql_query("SELECT ts, open, high, low, close, adj_close, volume, ticker FROM ohlc_hw_cleaned",
                             sqlite3.connect(Path.cwd() / 'data' / 'data.db'), parse_dates=['ts'])
after=generate_report(validate)

comparison = pd.DataFrame({'before': before['issue_type'].value_counts(),
                           'after': after['issue_type'].value_counts()}).fillna(0).astype(int)
comparison['change'] = comparison['after'] - comparison['before']
display(comparison)


adj_close cleaned: 28796, unresolved: 0
0 price data cleaned, 0 volume data cleaned
5 Non-positive values cleaned
0 Impossible OHLC px cleaned
5 Wrong OHLC px order cleaned
5056 extreme returns: 1418 market-wide, 2204 high-volume, 1434 unexplained
8 returns more than 10 s.d. from mean:


,ticker,ts,close,volume,extreme_category,prev_close,next_close,z
193606,JPM,2009-01-21,22.63000,136495312,unexplained,18.09000,23.10000,11.94
318332,TRV,1976-10-28,9.22000,0,unexplained,4.61000,4.69000,41.24
318333,TRV,1976-10-29,4.69000,79200,unexplained,9.22000,4.66000,20.30
318873,TRV,1978-12-20,8.03000,0,unexplained,4.02000,4.09000,41.14
318874,TRV,1978-12-21,4.09000,105600,unexplained,8.03000,4.09000,20.27
319538,TRV,1981-08-10,12.06000,0,unexplained,6.03000,6.03000,41.24
319539,TRV,1981-08-11,6.03000,24800,unexplained,12.06000,6.13000,20.66
331071,UNH,1987-05-04,0.19922,8681600,unexplained,0.15625,0.20703,10.98


3 rows interpolated from neighbouring days
saved 368251 rows to ohlc_hw_cleaned in c:\Users\adaka\github\UCB-MFE-python-preprogram-2026-winter\Homeworks\HW2\Ada_Kam\data\data.db


,before,after,change
issue_type,,,
Extreme Returns,5056,5175,119
Inconsistent Adj Close,10422,3,-10419
Missing Data,18374,0,-18374
Non Positive Data,5,0,-5
Verify Order,5,0,-5
Zero Volume,75,75,0
